<a href="https://colab.research.google.com/github/nuuuuurinn/Assessing-Racial-Disparities-in-Maternity-Care-with-Machine-Learning/blob/main/Final_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
#load imports
import os
import zipfile
from pathlib import Path
import shutil
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    f1_score,
    recall_score,
    precision_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

In [19]:
#load data
DATA_FILE = "DATASET.csv"

# Where the metrics and split datasets will be saved
WORK_DIR = Path("./Finalproject")
OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not Path(DATA_FILE).exists():
    raise FileNotFoundError(
        f"❌ Could not find {DATA_FILE}. Please ensure it is saved in the same folder as this script!"
    )

print(f"Loading {DATA_FILE}...")
final_df = pd.read_csv(DATA_FILE)

print("✅ Data loaded successfully!")
print("Shape:", final_df.shape)

Loading DATASET.csv...
✅ Data loaded successfully!
Shape: (363844, 10)


In [20]:
#separate dataset into train, test and validation
X = final_df.drop(columns=["SMM"])
y = final_df["SMM"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.25,   # 0.25 of 0.80 = 0.20 total validation
    random_state=42,
    stratify=y_train
)

train_df = X_train.copy()
train_df["SMM"] = y_train.values

val_df = X_val.copy()
val_df["SMM"] = y_val.values

test_df = X_test.copy()
test_df["SMM"] = y_test.values

train_csv = OUTPUT_DIR / "train_data.csv"
val_csv = OUTPUT_DIR / "validation_data.csv"
test_csv = OUTPUT_DIR / "test_data.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("\n✅ Saved split datasets:")
print(train_csv)
print(val_csv)
print(test_csv)


✅ Saved split datasets:
Finalproject/outputs/train_data.csv
Finalproject/outputs/validation_data.csv
Finalproject/outputs/test_data.csv


In [21]:
#processing
numeric_features = [
    "year",
    "maternal_age",
    "wic_used",
    "obesity",
    "multiple_gestations",
    "hypertensive_disease",
    "comorbidity_count"
]

categorical_features = [
    "insurance_status",
    "education"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()) #for lasso
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [22]:
#handling models with class imbalance
models = {
    "LogisticRegression_balanced": LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ),

    "Lasso_Classifier": LogisticRegression(
        penalty="l1",
        solver="liblinear",
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "RandomForest_balanced": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

results = []

for model_name, model in models.items():
    clf = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    clf.fit(X_train, y_train)

    val_pred = clf.predict(X_val)

    if hasattr(clf.named_steps["model"], "predict_proba"):
        val_scores = clf.predict_proba(X_val)[:, 1]
    else:
        val_scores = clf.decision_function(X_val)

    f1 = f1_score(y_val, val_pred, zero_division=0)
    recall = recall_score(y_val, val_pred, zero_division=0)
    precision = precision_score(y_val, val_pred, zero_division=0)
    auprc = average_precision_score(y_val, val_scores)

    results.append({
        "model": model_name,
        "f1": f1,
        "recall": recall,
        "precision": precision,
        "auprc": auprc,
        "pipeline": clf
    })

results_df = pd.DataFrame(results).drop(columns=["pipeline"]).sort_values(
    by=["auprc", "f1", "recall"],
    ascending=False
)

print("\n✅ Validation results:")
print(results_df)

best_model_name = results_df.iloc[0]["model"]
best_pipeline = next(r["pipeline"] for r in results if r["model"] == best_model_name)


✅ Validation results:
                         model        f1    recall  precision     auprc
1             Lasso_Classifier  0.038869  0.567593   0.020123  0.022017
0  LogisticRegression_balanced  0.038737  0.566667   0.020054  0.021999
2        RandomForest_balanced  0.038184  0.441667   0.019955  0.021155


In [23]:
#final test
test_pred = best_pipeline.predict(X_test)

if hasattr(best_pipeline.named_steps["model"], "predict_proba"):
    test_scores = best_pipeline.predict_proba(X_test)[:, 1]
else:
    test_scores = best_pipeline.decision_function(X_test)

test_f1 = f1_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_auprc = average_precision_score(y_test, test_scores)
test_cm = confusion_matrix(y_test, test_pred)
test_report = classification_report(y_test, test_pred, zero_division=0)

print("\n✅ Best model:", best_model_name)
print("Test F1:", test_f1)
print("Test Recall:", test_recall)
print("Test Precision:", test_precision)
print("Test AUPRC:", test_auprc)
print("\nConfusion Matrix:\n", test_cm)
print("\nClassification Report:\n", test_report)

#save metrics!
metrics_file = OUTPUT_DIR / "model_metrics.txt"

with open(metrics_file, "w", encoding="utf-8") as f:
    f.write("CDC 2024 Natality Modeling Results\n")
    f.write("=" * 50 + "\n\n")
    f.write("Target definition:\n")
    f.write("SMM = 1 if any of MM_MTR, MM_PLAC, MM_RUPT, MM_UHYST, MM_AICU == Y\n\n")

    f.write("Validation results:\n")
    f.write(results_df.to_string(index=False))
    f.write("\n\n")

    f.write(f"Best model: {best_model_name}\n\n")
    f.write(f"Test F1: {test_f1:.6f}\n")
    f.write(f"Test Recall: {test_recall:.6f}\n")
    f.write(f"Test Precision: {test_precision:.6f}\n")
    f.write(f"Test AUPRC: {test_auprc:.6f}\n\n")

    f.write("Confusion Matrix:\n")
    f.write(str(test_cm))
    f.write("\n\n")

    f.write("Classification Report:\n")
    f.write(test_report)

print("\n✅ Saved metrics file:")
print(metrics_file)

print("\n🎉 DONE — All outputs are in:")
print(OUTPUT_DIR)


✅ Best model: Lasso_Classifier
Test F1: 0.03614609571788413
Test Recall: 0.5314814814814814
Test Precision: 0.018709256844850065
Test AUPRC: 0.020766030181421365

Confusion Matrix:
 [[41583 30106]
 [  506   574]]

Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.58      0.73     71689
           1       0.02      0.53      0.04      1080

    accuracy                           0.58     72769
   macro avg       0.50      0.56      0.38     72769
weighted avg       0.97      0.58      0.72     72769


✅ Saved metrics file:
Finalproject/outputs/model_metrics.txt

🎉 DONE — All outputs are in:
Finalproject/outputs
